In [ ]:
!git clone https://github.com/iconrealestate77/thebeaverschoice.git
%cd thebeaverschoice
!pip install pandas numpy sqlalchemy python-dotenv smolagents -q

fatal: destination path 'thebeaverschoice' already exists and is not an empty directory.
/content/thebeaverschoice


In [ ]:
import os
from dotenv import load_dotenv

# Loads UDACITY_OPENAI_API_KEY from a local .env file (not committed to git).
# Create a .env file alongside this notebook containing:
#   UDACITY_OPENAI_API_KEY=your-key-here
load_dotenv()
assert os.environ.get("UDACITY_OPENAI_API_KEY"), (
    "UDACITY_OPENAI_API_KEY not found - add it to a .env file before running."
)

In [ ]:
!pwd
!ls -la

/content/thebeaverschoice
total 152
drwxr-xr-x 4 root root  4096 Aug 12 18:57 .
drwxr-xr-x 1 root root  4096 Aug 12 18:54 ..
drwxr-xr-x 8 root root  4096 Aug 12 18:55 .git
-rw-r--r-- 1 root root 28491 Aug 12 18:55 project_starter.py
-rw-r--r-- 1 root root 30684 Aug 12 18:55 quote_requests.csv
-rw-r--r-- 1 root root  5825 Aug 12 18:55 quote_requests_sample.csv
-rw-r--r-- 1 root root 57510 Aug 12 18:55 quotes.csv
-rw-r--r-- 1 root root  3584 Aug 12 18:55 README.md
-rw-r--r-- 1 root root    84 Aug 12 18:55 requirements.txt
drwxr-xr-x 3 root root  4096 Aug 12 18:57 thebeaverschoice


In [ ]:
%cd project
!cat requirements.txt

/content/thebeaverschoice/project
pandas==2.2.3
typing==3.7.4.3
openai==1.76.0
SQLAlchemy==2.0.40
python-dotenv==1.1.0

In [ ]:
!pip install -r requirements.txt -q
!pip install smolagents -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 103.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.2/661.2 kB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 89.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.2.3 which is incompatible.


In [ ]:
from dotenv import load_dotenv

# Re-load in case this cell runs in a fresh kernel/session after the %cd above.
load_dotenv()
assert os.environ.get("UDACITY_OPENAI_API_KEY"), (
    "UDACITY_OPENAI_API_KEY not found - add it to a .env file before running."
)

In [ ]:
import project_starter as pb

pb.init_database(pb.db_engine)
print("DB initialized OK")

report = pb.generate_financial_report("2025-04-01")
print(report)

DB initialized OK
{'as_of_date': '2025-04-01', 'cash_balance': 45059.7, 'inventory_value': np.float64(4940.299999999999), 'total_assets': np.float64(50000.0), 'inventory_summary': [{'item_name': 'Paper plates', 'stock': np.float64(748.0), 'unit_price': 0.1, 'value': np.float64(74.8)}, {'item_name': '100 lb cover stock', 'stock': np.float64(636.0), 'unit_price': 0.5, 'value': np.float64(318.0)}, {'item_name': 'Glossy paper', 'stock': np.float64(587.0), 'unit_price': 0.2, 'value': np.float64(117.4)}, {'item_name': 'Rolls of banner paper (36-inch width)', 'stock': np.float64(546.0), 'unit_price': 2.5, 'value': np.float64(1365.0)}, {'item_name': 'Photo paper', 'stock': np.float64(423.0), 'unit_price': 0.25, 'value': np.float64(105.75)}, {'item_name': 'Cardstock', 'stock': np.float64(595.0), 'unit_price': 0.15, 'value': np.float64(89.25)}, {'item_name': 'Colored paper', 'stock': np.float64(788.0), 'unit_price': 0.1, 'value': np.float64(78.80000000000001)}, {'item_name': '80 lb text paper', 

In [ ]:
import re

STOPWORDS = {"paper", "sheets", "sheet", "of", "the", "a", "an"}

def _tokenize(name: str) -> set:
    return set(re.findall(r"[a-zA-Z0-9]+", name.lower()))

def _size_token(name: str) -> str | None:
    match = re.search(r"\bA[3-6]\b", name, re.IGNORECASE)
    return match.group(0).upper() if match else None

CATALOG_NAMES = [item["item_name"] for item in pb.paper_supplies]

def match_catalog_item(requested_name: str, min_overlap: float = 0.5) -> str | None:
    """
    Match a free-text item name to the closest item in our paper_supplies catalog
    using descriptive-word overlap rather than character similarity. Explicitly
    rejects size mismatches (e.g. A3 vs A4) so we never silently substitute a
    different product than what was requested.

    Args:
        requested_name: the raw item name as written by the customer
        min_overlap: minimum fraction of the candidate's descriptive words that
                     must appear in the request for it to count as a match

    Returns:
        The matching catalog item name, or None if nothing matches closely enough.
    """
    requested_size = _size_token(requested_name)
    requested_tokens = _tokenize(requested_name) - STOPWORDS

    best_match, best_score = None, 0.0

    for candidate in CATALOG_NAMES:
        candidate_size = _size_token(candidate)

        # Reject if request specifies a size our candidate doesn't have, or vice versa
        if requested_size != candidate_size:
            continue

        candidate_tokens = _tokenize(candidate) - STOPWORDS
        if not candidate_tokens:
            continue

        overlap = requested_tokens.intersection(candidate_tokens)
        score = len(overlap) / len(candidate_tokens)

        if score > best_score:
            best_score, best_match = score, candidate

    return best_match if best_score >= min_overlap else None

In [ ]:
print(match_catalog_item("heavy cardstock"))
print(match_catalog_item("A3 paper"))
print(match_catalog_item("glossy A4 paper"))
print(match_catalog_item("A4 paper"))
print(match_catalog_item("printer paper"))
print(match_catalog_item("standard printer paper"))

Cardstock
None
A4 paper
A4 paper
None
Standard copy paper


In [ ]:
from smolagents import tool

@tool
def check_inventory(item_name: str, as_of_date: str) -> dict:
    """
    Check current stock level for a specific paper/product item as of a given date.
    Automatically matches free-text item names to the closest item in our catalog.

    Args:
        item_name: the name of the item to check (can be approximate/free-text)
        as_of_date: ISO date string (YYYY-MM-DD) for the stock snapshot

    Returns:
        A dict with matched_item, current_stock, unit_price, and found (bool).
        If found is False, this item is not carried in our catalog.
    """
    matched = match_catalog_item(item_name)
    if not matched:
        return {"matched_item": None, "current_stock": 0, "unit_price": None, "found": False}

    stock_df = pb.get_stock_level(matched, as_of_date)
    current_stock = int(stock_df["current_stock"].iloc[0]) if not stock_df.empty else 0

    unit_price = next(
        (p["unit_price"] for p in pb.paper_supplies if p["item_name"] == matched), None
    )

    return {
        "matched_item": matched,
        "current_stock": current_stock,
        "unit_price": unit_price,
        "found": True,
    }


@tool
def check_delivery_date(as_of_date: str, quantity: int) -> str:
    """
    Estimate the delivery date from a supplier for a given order quantity, starting from a date.

    Args:
        as_of_date: ISO date string (YYYY-MM-DD) representing the order date
        quantity: number of units being ordered

    Returns:
        Estimated delivery date as an ISO date string (YYYY-MM-DD).
    """
    return pb.get_supplier_delivery_date(as_of_date, quantity)

In [ ]:
print(check_inventory("cardstock", "2025-04-01"))
print(check_inventory("A3 paper", "2025-04-01"))

{'matched_item': 'Cardstock', 'current_stock': 595, 'unit_price': 0.15, 'found': True}
{'matched_item': None, 'current_stock': 0, 'unit_price': None, 'found': False}


In [ ]:
@tool
def search_past_quotes(search_terms: list[str], limit: int = 5) -> list[dict]:
    """
    Search historical quotes for similar past requests, useful for pricing consistency.

    Args:
        search_terms: list of keywords to search for (e.g. item names, event type, job type)
        limit: maximum number of past quotes to return

    Returns:
        A list of matching past quotes with original_request, total_amount,
        quote_explanation, job_type, order_size, event_type, and order_date.
    """
    return pb.search_quote_history(search_terms, limit=limit)


@tool
def calculate_quote(item_name: str, quantity: int, unit_price: float) -> dict:
    """
    Calculate a price quote for a given item and quantity, applying bulk discounts.

    Discount tiers (applied to the line total):
        - 1000+ units: 15% off
        - 500-999 units: 10% off
        - 200-499 units: 5% off
        - under 200 units: no discount

    Args:
        item_name: the catalog item name being quoted
        quantity: number of units requested
        unit_price: price per unit from the catalog

    Returns:
        A dict with item_name, quantity, unit_price, subtotal, discount_pct,
        discount_amount, and line_total.
    """
    subtotal = quantity * unit_price

    if quantity >= 1000:
        discount_pct = 0.15
    elif quantity >= 500:
        discount_pct = 0.10
    elif quantity >= 200:
        discount_pct = 0.05
    else:
        discount_pct = 0.0

    discount_amount = subtotal * discount_pct
    line_total = subtotal - discount_amount

    return {
        "item_name": item_name,
        "quantity": quantity,
        "unit_price": unit_price,
        "subtotal": round(subtotal, 2),
        "discount_pct": discount_pct,
        "discount_amount": round(discount_amount, 2),
        "line_total": round(line_total, 2),
    }

In [ ]:
print(calculate_quote("Cardstock", 300, 0.15))
print(calculate_quote("A4 paper", 10000, 0.05))
print(search_past_quotes(["ceremony", "cardstock"], limit=3))

{'item_name': 'Cardstock', 'quantity': 300, 'unit_price': 0.15, 'subtotal': 45.0, 'discount_pct': 0.05, 'discount_amount': 2.25, 'line_total': 42.75}
{'item_name': 'A4 paper', 'quantity': 10000, 'unit_price': 0.05, 'subtotal': 500.0, 'discount_pct': 0.15, 'discount_amount': 75.0, 'line_total': 425.0}
[{'original_request': 'I would like to place an order for 500 sheets of high-quality white cardstock and 1000 sheets of colored printer paper for our upcoming ceremony. Please ensure delivery by April 15, 2025. Thank you.', 'total_amount': 160, 'quote_explanation': "Thank you for your order! For 500 sheets of high-quality white cardstock, we typically charge $0.15 each, bringing the subtotal to $75. For the 1000 sheets of colored printer paper at $0.10 each, the subtotal is $100. To help make your order more budget-friendly, I'm happy to apply a bulk discount, rounding the total for both items down to $160, which is a more manageable figure. We will ensure delivery by April 15, 2025.", 'jo

In [ ]:
import pandas as pd

@tool
def check_finances(as_of_date: str) -> dict:
    """
    Get the company's cash balance and a full financial report as of a given date.
    Values ALL catalog items that have ever had a transaction, not just the items
    in the original seeded inventory snapshot (pb.generate_financial_report only
    looks at that snapshot, which under-reports inventory value for items ordered
    outside the initial 40% coverage).

    Args:
        as_of_date: ISO date string (YYYY-MM-DD)

    Returns:
        A dict with cash_balance, inventory_value, total_assets, and inventory_summary.
    """
    cash = pb.get_cash_balance(as_of_date)

    # Every item that has ever had a transaction, not just the seeded inventory table
    item_names = pd.read_sql(
        "SELECT DISTINCT item_name FROM transactions WHERE item_name IS NOT NULL",
        pb.db_engine,
    )["item_name"].tolist()

    price_lookup = {p["item_name"]: p["unit_price"] for p in pb.paper_supplies}

    inventory_value = 0.0
    inventory_summary = []
    for item_name in item_names:
        stock_df = pb.get_stock_level(item_name, as_of_date)
        stock = int(stock_df["current_stock"].iloc[0]) if not stock_df.empty else 0
        if stock <= 0:
            continue
        unit_price = price_lookup.get(item_name)
        if unit_price is None:
            continue  # not a real catalog item, skip valuation
        value = stock * unit_price
        inventory_value += value
        inventory_summary.append({
            "item_name": item_name,
            "stock": stock,
            "unit_price": unit_price,
            "value": round(value, 2),
        })

    return {
        "as_of_date": as_of_date,
        "cash_balance": round(cash, 2),
        "inventory_value": round(inventory_value, 2),
        "total_assets": round(cash + inventory_value, 2),
        "inventory_summary": inventory_summary,
    }


@tool
def record_sale(item_name: str, quantity: int, total_price: float, date: str) -> dict:
    """
    Record a finalized sale transaction in the company's database.

    Args:
        item_name: the catalog item name sold
        quantity: number of units sold
        total_price: total price charged for this line item
        date: ISO date string (YYYY-MM-DD) of the sale

    Returns:
        A dict with transaction_id and status.
    """
    transaction_id = pb.create_transaction(
        item_name=item_name,
        transaction_type="sales",
        quantity=quantity,
        price=total_price,
        date=date,
    )
    return {"transaction_id": transaction_id, "status": "recorded"}


@tool
def record_stock_order(item_name: str, quantity: int, total_price: float, date: str) -> dict:
    """
    Record a stock reorder transaction (purchasing more inventory from a supplier).

    Args:
        item_name: the catalog item name being reordered
        quantity: number of units ordered
        total_price: total cost of the reorder
        date: ISO date string (YYYY-MM-DD) of the order

    Returns:
        A dict with transaction_id and status.
    """
    transaction_id = pb.create_transaction(
        item_name=item_name,
        transaction_type="stock_orders",
        quantity=quantity,
        price=total_price,
        date=date,
    )
    return {"transaction_id": transaction_id, "status": "recorded"}

In [ ]:
print(check_finances("2025-04-01"))
result = record_sale("Cardstock", 5, 0.75, "2025-04-01")
print(result)
print(check_finances("2025-04-01"))

{'as_of_date': '2025-04-01', 'cash_balance': 45060.45, 'inventory_value': 4939.55, 'total_assets': 50000.0, 'inventory_summary': [{'item_name': 'Paper plates', 'stock': 748, 'unit_price': 0.1, 'value': 74.8}, {'item_name': '100 lb cover stock', 'stock': 636, 'unit_price': 0.5, 'value': 318.0}, {'item_name': 'Glossy paper', 'stock': 587, 'unit_price': 0.2, 'value': 117.4}, {'item_name': 'Rolls of banner paper (36-inch width)', 'stock': 546, 'unit_price': 2.5, 'value': 1365.0}, {'item_name': 'Photo paper', 'stock': 423, 'unit_price': 0.25, 'value': 105.75}, {'item_name': 'Cardstock', 'stock': 590, 'unit_price': 0.15, 'value': 88.5}, {'item_name': 'Colored paper', 'stock': 788, 'unit_price': 0.1, 'value': 78.8}, {'item_name': '80 lb text paper', 'stock': 249, 'unit_price': 0.4, 'value': 99.6}, {'item_name': 'Large poster paper (24x36 inches)', 'stock': 699, 'unit_price': 1.0, 'value': 699.0}, {'item_name': 'Table covers', 'stock': 736, 'unit_price': 1.5, 'value': 1104.0}, {'item_name': 'B

In [ ]:
from smolagents import ToolCallingAgent, CodeAgent, OpenAIServerModel

model = OpenAIServerModel(
    model_id="gpt-4o-mini",
    api_base="https://openai.vocareum.com/v1",
    api_key=os.environ["UDACITY_OPENAI_API_KEY"],
)

inventory_agent = ToolCallingAgent(
    tools=[check_inventory, check_delivery_date],
    model=model,
    name="inventory_agent",
    max_steps=5,
    description=(
        "Checks current stock levels for requested items and estimates supplier "
        "delivery dates for reorders. Use this to find out if we have enough stock "
        "of an item, or when more stock would arrive if we don't. "
        "IMPORTANT: only check inventory as of the ONE exact date you are given — "
        "never check multiple different dates or date ranges. If you are not "
        "explicitly given a date in your task, do NOT guess one — report back to "
        "your manager that you need the request date."
    ),
)

quoting_agent = ToolCallingAgent(
    tools=[search_past_quotes, calculate_quote],
    model=model,
    name="quoting_agent",
    max_steps=5,
    description=(
        "Generates a price quote for a SPECIFIC item and quantity, applying bulk "
        "discounts and referencing historical quote pricing for consistency. "
        "You MUST call this for every item in the order to get its price BEFORE "
        "any sale can be finalized. ONLY quote items that inventory_agent has "
        "confirmed with found=True. Do NOT quote an item just because "
        "search_past_quotes returns historical quotes mentioning a similar name — "
        "those are unrelated past orders, not proof the item is currently sellable."
    ),
)

sales_agent = ToolCallingAgent(
    tools=[check_finances, record_sale, record_stock_order],
    model=model,
    name="sales_agent",
    max_steps=5,
    description=(
        "Finalizes sales transactions by recording them in the database (which "
        "changes the company's cash balance), checks company cash balance and "
        "financial health, and records stock reorders. You MUST call this to "
        "actually complete/record an order after inventory and pricing are confirmed — "
        "an order is NOT complete until sales_agent has recorded it. "
        "IMPORTANT: every check_finances, record_sale, and record_stock_order call "
        "MUST use the EXACT request date you are given in your task — never invent, "
        "assume, or default to any other date (including today's date). If you are "
        "not explicitly given a date in your task, do NOT guess one — report back "
        "to your manager that you need the request date. NEVER record a sale for an "
        "item that was not confirmed as found=True by inventory_agent."
    ),
)

ORCHESTRATOR_INSTRUCTIONS = """
You are the orchestrator for Beaver's Choice Paper Company's order fulfillment system.

CRITICAL: Managed agents (inventory_agent, quoting_agent, sales_agent) take exactly
ONE argument: a single task string. Call them like inventory_agent("task text here") --
NEVER pass a second positional argument (e.g. NEVER call
inventory_agent("task text", {"item": item}) or inventory_agent(task, extra_dict)).
If you need to pass extra context, put it directly inside the task string itself.

CRITICAL: The customer request below includes a specific request date. You MUST use
this EXACT date (and no other date) for every check_inventory, check_delivery_date,
check_finances, and record_sale/record_stock_order call. NEVER invent, assume, or
default to any other date, including today's date or any date from your own training.
Always pass the date explicitly, as a literal string, into EVERY SINGLE task you give
to a managed agent, including follow-up calls within the same order — do not rely on
the managed agent to infer or remember it. NEVER ask a managed agent to check multiple
dates or date ranges — only ever the one exact request date.

For EVERY customer request, you MUST follow this exact process, in order:

1. Call inventory_agent to check stock levels for EVERY item requested, and get
   delivery date estimates for any items that need reordering. Explicitly state
   the request date in the task you give inventory_agent.

2. For EVERY item that IS in our catalog (found=True) and has EITHER enough stock
   OR an acceptable delivery timeline before the customer's needed-by date, call
   quoting_agent to get a price quote for that item and quantity.

   CRITICAL: If inventory_agent reported found=False for an item, that item is NOT
   in our catalog. You MUST NOT call quoting_agent or sales_agent for it under any
   circumstances, even if quoting_agent's search_past_quotes tool returns historical
   quotes that happen to mention similar item names — those are unrelated past orders,
   not proof the item is currently sellable. A found=False item is ALWAYS rejected
   at step 3, with no further action.

3. If ANY of these conditions apply, the order (or that specific line item) CANNOT
   be fulfilled and must be REJECTED with a clear reason:
   - The item is not in our catalog (found=False)
   - Stock is insufficient AND the supplier delivery date is after the customer's
     needed-by date
   - The company's cash balance (check via sales_agent, passing the request date
     explicitly) is insufficient to justify a large reorder

4. For every item that CAN be fulfilled (found=True AND passes the checks above),
   call sales_agent to record the sale transaction (this is what actually finalizes
   the order and updates cash balance). Use the SAME exact request date for this
   transaction. If a reorder is needed to fulfill demand, also record a stock_order
   transaction via sales_agent before recording the sale.

5. Compose a final customer-facing response that:
   - Lists each item, whether it was fulfilled or rejected, and why
   - States the price for each fulfilled item and the total order price
   - States the expected delivery date
   - Does NOT reveal internal details like exact profit margins or raw error messages

You MUST NOT skip steps 2 and 4 for fulfillable items. An order is only complete
once sales_agent has recorded the transaction(s). You MUST NEVER record a sale or
generate a quote for an item that inventory_agent reported as found=False.
"""

orchestrator = CodeAgent(
    tools=[],
    model=model,
    managed_agents=[inventory_agent, quoting_agent, sales_agent],
    name="orchestrator",
    description="Manages the full customer order workflow for Beaver's Choice Paper Company.",
    max_steps=12,  # bumped from 8: extra headroom while the agent settles into the
                   # single-argument managed-agent calling convention above
)

In [ ]:
def run_orchestrator(customer_request: str, request_date: str) -> str:
    full_task = (
        f"{ORCHESTRATOR_INSTRUCTIONS}\n\n"
        f"---\n"
        f"REQUEST DATE (use this exact date for all tool calls): {request_date}\n\n"
        f"Customer request:\n{customer_request}\n\n"
        f"REMINDER: request date is {request_date}. Use it for every tool call."
    )
    result = orchestrator.run(full_task)
    return result

In [ ]:
response = run_orchestrator(
    "I would like to request the following paper supplies for the ceremony: "
    "- 200 sheets of A4 glossy paper - 100 sheets of heavy cardstock (white) - "
    "100 sheets of colored paper (assorted colors) I need these supplies delivered "
    "by April 15, 2025. Thank you.",
    "2025-04-01",
)
print(response)

╭──────────────────────────────────────────── New run - orchestrator ─────────────────────────────────────────────╮
│                                                                                                                 │
│ You are the orchestrator for Beaver's Choice Paper Company's order fulfillment system.                          │
│                                                                                                                 │
│ CRITICAL: Managed agents (inventory_agent, quoting_agent, sales_agent) take exactly                             │
│ ONE argument: a single task string. Call them like inventory_agent("task text here") --                         │
│ NEVER pass a second positional argument (e.g. NEVER call                                                        │
│ inventory_agent("task text", {"item": item}) or inventory_agent(task, extra_dict)).                             │
│ If you need to pass extra context, put it directly inside the task string itself.                               │
│                                                                                                                 │
│ CRITICAL: The customer request below includes a specific request date. You MUST use                             │
│ this EXACT date (and no other date) for every check_inventory, check_delivery_date,                             │
│ check_finances, and record_sale/record_stock_order call. NEVER invent, assume, or                               │
│ default to any other date, including today's date or any date from your own training.                           │
│ Always pass the date explicitly, as a literal string, into EVERY SINGLE task you give                           │
│ to a managed agent, including follow-up calls within the same order — do not rely on                            │
│ the managed agent to infer or remember it. NEVER ask a managed agent to check multiple                          │
│ dates or date ranges — only ever the one exact request date.                                                    │
│                                                                                                                 │
│ For EVERY customer request, you MUST follow this exact process, in order:                                       │
│                                                                                                                 │
│ 1. Call inventory_agent to check stock levels for EVERY item requested, and get                                 │
│    delivery date estimates for any items that need reordering. Explicitly state                                 │
│    the request date in the task you give inventory_agent.                                                       │
│                                                                                                                 │
│ 2. For EVERY item that IS in our catalog (found=True) and has EITHER enough stock                               │
│    OR an acceptable delivery timeline before the customer's needed-by date, call                                │
│    quoting_agent to get a price quote for that item and quantity.                                               │
│                                                                                                                 │
│    CRITICAL: If inventory_agent reported found=False for an item, that item is NOT                              │
│    in our catalog. You MUST NOT call quoting_agent or sales_agent for it under any                              │
│    circumstances, even if quoting_agent's search_past_quotes tool returns historical                            │
│    quotes that happen to mention similar item names — those are unrelated past orders,                          │
│    not proof the item is currently sellable. A found=False item is ALWAYS rejected                              │
│    at step 3, with no further action.                 

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  inventory_check_results = []                                                                                     
  inventory_check_results.append(inventory_agent("Check stock levels for 200 sheets of A4 glossy paper on the      
  request date 2025-04-01."))                                                                                      
  inventory_check_results.append(inventory_agent("Check stock levels for 100 sheets of heavy cardstock (white) on  
  the request date 2025-04-01."))                                                                                  
  inventory_check_results.append(inventory_agent("Check stock levels for 100 sheets of colored paper (assorted     
  colors) on the request date 2025-04-01."))                                                                       
  print(inventory_check_results)                                                                                   
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

╭─────────────────────────────────────────── New run - inventory_agent ───────────────────────────────────────────╮
│                                                                                                                 │
│ You're a helpful agent named 'inventory_agent'.                                                                 │
│ You have been submitted this task by your manager.                                                              │
│ ---                                                                                                             │
│ Task:                                                                                                           │
│ Check stock levels for 200 sheets of A4 glossy paper on the request date 2025-04-01.                            │
│ ---                                                                                                             │
│ You're helping your manager solve a wider task: so make sure to not provide a one-line answer, but give as much │
│ information as possible to give them a clear understanding of the answer.                                       │
│                                                                                                                 │
│ Your final_answer WILL HAVE to contain these parts:                                                             │
│ ### 1. Task outcome (short version):                                                                            │
│ ### 2. Task outcome (extremely detailed version):                                                               │
│ ### 3. Additional context (if relevant):                                                                        │
│                                                                                                                 │
│ Put all these in your final_answer tool, everything that you do not pass as an argument to final_answer will be │
│ lost.                                                                                                           │
│ And even if your task resolution is not successful, please return as much context as possible, so that your     │
│ manager can act upon this feedback.                                                                             │
│                                                                                                                 │
╰─ OpenAIModel - gpt-4o-mini ─────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 5 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_response = (                                                                                               
      "Dear Customer,\n\n"                                                                                         
      "Thank you for your order! Here are the details:\n\n"                                                        
      "- 200 sheets of A4 glossy paper: Fulfilled at $19.00\n"                                                     
      "- 100 sheets of heavy cardstock (white): Fulfilled at $50.00\n"                                             
      "- 100 sheets of colored paper (assorted colors): Fulfilled at $10.00\n\n"                                   
      "Total Order Price: $79.00\n"                                                                                
      "Expected Delivery Date: April 15, 2025\n\n"                                                                 
      "Thank you for choosing Beaver's Choice Paper Company!"                                                      
  )                                                                                                                
                                                                                                                   
  final_answer(final_response)                                                                                     
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Dear Customer,

Thank you for your order! Here are the details:

- 200 sheets of A4 glossy paper: Fulfilled at $19.00
- 100 sheets of heavy cardstock (white): Fulfilled at $50.00
- 100 sheets of colored paper (assorted colors): Fulfilled at $10.00

Total Order Price: $79.00
Expected Delivery Date: April 15, 2025

Thank you for choosing Beaver's Choice Paper Company!

[Step 5: Duration 6.79 seconds| Input tokens: 28,053 | Output tokens: 1,613]

Dear Customer,

Thank you for your order! Here are the details:

- 200 sheets of A4 glossy paper: Fulfilled at $19.00
- 100 sheets of heavy cardstock (white): Fulfilled at $50.00
- 100 sheets of colored paper (assorted colors): Fulfilled at $10.00

Total Order Price: $79.00
Expected Delivery Date: April 15, 2025

Thank you for choosing Beaver's Choice Paper Company!


In [ ]:
print(pb.generate_financial_report("2025-04-01")["cash_balance"])

45164.7


In [ ]:
import time

def run_test_scenarios():
    print("Initializing Database...")
    pb.init_database(pb.db_engine)

    try:
        quote_requests_sample = pd.read_csv("quote_requests_sample.csv")
        quote_requests_sample["request_date"] = pd.to_datetime(
            quote_requests_sample["request_date"], format="%m/%d/%y", errors="coerce"
        )
        quote_requests_sample.dropna(subset=["request_date"], inplace=True)
        quote_requests_sample = quote_requests_sample.sort_values("request_date")
    except Exception as e:
        print(f"FATAL: Error loading test data: {e}")
        return

    initial_date = quote_requests_sample["request_date"].min().strftime("%Y-%m-%d")
    # Use check_finances (not pb.generate_financial_report) so inventory value covers
    # every item ever transacted, not just the original seeded 40% inventory snapshot.
    report = check_finances(initial_date)
    current_cash = report["cash_balance"]
    current_inventory = report["inventory_value"]

    results = []
    for idx, row in quote_requests_sample.iterrows():
        request_date = row["request_date"].strftime("%Y-%m-%d")

        print(f"\n=== Request {idx+1} ===")
        print(f"Context: {row['job']} organizing {row['event']}")
        print(f"Request Date: {request_date}")
        print(f"Cash Balance: ${current_cash:.2f}")
        print(f"Inventory Value: ${current_inventory:.2f}")

        request_with_date = f"{row['request']} (Date of request: {request_date})"

        try:
            response = run_orchestrator(request_with_date, request_date)
        except Exception as e:
            response = f"ERROR processing request: {e}"
            print(response)

        report = check_finances(request_date)
        current_cash = report["cash_balance"]
        current_inventory = report["inventory_value"]

        print(f"Response: {response}")
        print(f"Updated Cash: ${current_cash:.2f}")
        print(f"Updated Inventory: ${current_inventory:.2f}")

        results.append(
            {
                "request_id": idx + 1,
                "request_date": request_date,
                "cash_balance": current_cash,
                "inventory_value": current_inventory,
                "response": response,
            }
        )

        time.sleep(1)

    final_date = quote_requests_sample["request_date"].max().strftime("%Y-%m-%d")
    final_report = check_finances(final_date)
    print("\n===== FINAL FINANCIAL REPORT =====")
    print(f"Final Cash: ${final_report['cash_balance']:.2f}")
    print(f"Final Inventory: ${final_report['inventory_value']:.2f}")

    pd.DataFrame(results).to_csv("test_results.csv", index=False)
    return results


In [ ]:
results = run_test_scenarios()

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
import pandas as pd
df = pd.read_csv("test_results.csv")
print(df.shape)
df[["request_id", "cash_balance", "inventory_value"]]

(20, 5)


,request_id,cash_balance,inventory_value
0,1,45157.70,4940.3
1,2,45525.20,4940.3
2,3,45525.20,4940.3
3,4,45683.96,4940.3
4,5,46013.96,4940.3
5,6,46042.46,4940.3
6,7,46232.46,4940.3
7,8,46232.46,4940.3
8,9,46241.96,4940.3
9,13,46526.96,4940.3
